[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/47_moe_core.ipynb)

# 🔴 Hard: MoE Core (Simplified Dense Routing)

Implement a simplified **MoE core** where every token uses **all experts** with soft routing.

### Signature
```python
class SimpleMoECore(nn.Module):
    def __init__(self, d_model, num_experts): ...
    def forward(self, x):
        # x: (B, S, D) -> (B, S, D)
```

### Idea
- Router logits: `self.router(x)` -> shape `(B, S, E)`
- Weights: `softmax(logits, dim=-1)`
- Run each expert on `x`, stack outputs `(B, S, E, D)`
- Weighted sum over experts

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class SimpleMoECore(nn.Module):
    def __init__(self, d_model, num_experts):
        super().__init__()
        self.router=nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([ nn.Linear(d_model, d_model) for _ in range(num_experts) ])

    def forward(self, x):
        route_logits = self.router(x) # b,s,e
        weights = torch.softmax(route_logits, dim=-1) # 注意是先变为概率
        expert_out = [ expert(x) for expert in self.experts]
        expert_out = torch.stack(expert_out, dim=-2)#.transpose(-1,-2) # b,s,e,d
        print(weights.shape, expert_out.shape)
        out = weights.unsqueeze(-1) * expert_out
        out = out.sum(dim=-2) # 因为是概率加权，所以是sum
        return out


In [9]:
# 🧪 Debug
moe = SimpleMoECore(d_model=16, num_experts=4)
x = torch.randn(2, 6, 16)
out = moe(x)
print('Output shape:', out.shape)

torch.Size([2, 6, 4]) torch.Size([2, 6, 4, 16])
Output shape: torch.Size([2, 6, 16])


In [6]:
import sys
sys.path.append("/data/chenjuntao/OtherProj/TorchCode")

In [12]:
# ✅ SUBMIT

from torch_judge import check
check('moe_core')


🧪 Testing: MoE Core (Simplified Dense Routing) (Hard)
──────────────────────────────────────────────────
torch.Size([2, 8, 4]) torch.Size([2, 8, 4, 16])
  ✅ [1/4] Output shape (4.1ms)
  ✅ [2/4] Has router and experts (1.4ms)
torch.Size([1, 2, 3]) torch.Size([1, 2, 3, 4])
  ✅ [3/4] Weighted sum matches manual reference (4.2ms)
torch.Size([2, 5, 4]) torch.Size([2, 5, 4, 12])
  ✅ [4/4] Gradient flow (4.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (14.5ms total)
  Progress saved. Run status() to see your dashboard.

